In [ ]:
'''!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("snir5").project("almons-trees")
version = project.version(7)
dataset = version.download("coco-segmentation")'''



from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("snir5").project("almons-trees")
version = project.version(8)
dataset = version.download("coco-segmentation")
                
                

In [ ]:
import cv2, numpy as np
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import glob
import random
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
def gray_world_wb(img):
    # img uint8 RGB
    imgf = img.astype(np.float32)
    mean = imgf.reshape(-1,3).mean(axis=0) + 1e-6
    scale = mean.mean() / mean
    out = np.clip(imgf * scale, 0, 255).astype(np.uint8)
    return out

def adaptive_gamma(img, target_v=0.5, clip=(0.7, 1.4)):
    # move average luminance toward target_v deterministically
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2].astype(np.float32)/255.0
    v_mean = float(np.clip(v.mean(), 0.05, 0.95))
    gamma = np.log(v_mean) / np.log(max(target_v, 1e-6))
    gamma = float(np.clip(gamma, clip[0], clip[1]))
    x = (img.astype(np.float32)/255.0) ** (1.0/gamma)
    return np.clip(x*255.0,0,255).astype(np.uint8)

def adaptive_clahe(img, base_clip=2.0, tile=(8,8)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2]
    v_std = float(v.std())/255.0
    # lower contrast -> stronger CLAHE; keep bounded
    clip_limit = float(np.clip(base_clip + (0.8 - v_std)*1.0, 1.5, 3.0))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    hsv[...,2] = clahe.apply(v)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def adaptive_color_deterministic(img):
    # 1) white balance  2) gamma to target  3) CLAHE by contrast
    wb  = gray_world_wb(img)
    gam = adaptive_gamma(wb, target_v=0.5, clip=(0.8, 1.3))
    out = adaptive_clahe(gam, base_clip=2.0, tile=(8,8))
    return out


In [ ]:
# Different normalization options
NORM_TECHNIQUES = {
    "imagenet": {
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225)
    },
    "dataset": None,  # Will compute from dataset
    "minmax": {"mean": (0.0, 0.0, 0.0), "std": (1.0, 1.0, 1.0)},
    "none": None
}

def compute_dataset_mean_std(img_dir, sample_size=500):
    """Compute mean and std for dataset (RGB in [0,1])"""
    import glob
    import cv2
    import numpy as np

    img_paths = glob.glob(os.path.join(img_dir, "*.jpg"))[:sample_size]
    means, stds = [], []

    for path in img_paths:
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))

    mean = np.mean(means, axis=0)
    std = np.mean(stds, axis=0)
    return tuple(mean), tuple(std)


def get_base_transform(norm_type="imagenet", dataset_img_dir=None):
    if norm_type == "dataset" and dataset_img_dir:
        mean, std = compute_dataset_mean_std(dataset_img_dir)
        print(f"📊 Dataset normalization: mean={mean}, std={std}")
        norm_transform = A.Normalize(mean=mean, std=std)
    elif norm_type == "imagenet":
        mean_std = NORM_TECHNIQUES["imagenet"]
        norm_transform = A.Normalize(mean=mean_std["mean"], std=mean_std["std"])
    elif norm_type == "minmax":
        norm_transform = A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0))
    elif norm_type == "none":
        norm_transform = A.Lambda(image=lambda x, **kwargs: x)  # no change
    else:
        raise ValueError(f"Unknown normalization type: {norm_type}")

    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Equalize(mode='cv', p=0.5),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.7, 1.0), p=0.4),
        norm_transform,
        ToTensorV2()
    ])


In [ ]:
IMAGE_SIZE = 1024

# Example: ImageNet (default for pretrained models)
train_transform = get_base_transform(norm_type="imagenet")
'''
# Example: Min-Max [0, 1] normalization
train_transform = get_base_transform(norm_type="minmax")

# Example: No normalization
train_transform = get_base_transform(norm_type="none")

# Example: Dataset-based normalization
train_transform = get_base_transform(norm_type="dataset", dataset_img_dir=train_img_dir)
'''

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import glob
import random
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# 📐 Constants
IMAGE_SIZE = 1024

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_val_test_transform_adaptive(IMAGE_SIZE=1024, norm="imagenet"):
    if norm == "imagenet":
        norm_tf = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    elif norm == "minmax":
        norm_tf = A.Normalize(mean=(0,0,0), std=(1,1,1))
    elif norm == "none":
        norm_tf = A.Lambda(image=lambda x, **k: x)
    else:
        raise ValueError("norm must be 'imagenet' | 'minmax' | 'none'")

    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Lambda(image=lambda x, **k: adaptive_color_deterministic(x)),  # deterministic, per-image
        norm_tf,
        ToTensorV2()
    ])

'''# 🧱 Base transform for *all* datasets — to enhance tree clarity consistently
def get_base_transform():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Equalize(mode='cv', p=0.5),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.7, 1.0), p=0.4),
        A.Normalize(),
        ToTensorV2()
    ])'''


# 🌱 Full transform for TRAINING dataset: base + tree-specific augmentations
def get_train_transform():
    base = get_base_transform()
    aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.3),
        A.Transpose(p=0.3),
        A.RandomGamma(gamma_limit=(60, 140), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=15, p=0.4),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),
        A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.6), p=0.3)
    ])
    return A.Compose(aug.transforms + base.transforms)  # Combine augmentations + base


class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = np.fliplr(image)  # Flip image

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))
        mask = np.fliplr(mask)  # Flip mask to match image

        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        augmented = self.transform(image=image, mask=mask)
        image = augmented['image']
        mask = (augmented['mask'] > 0).unsqueeze(0).float()
        return image, mask




# 🛠️ Set up paths
train_img_dir = "/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# 📦 Load transforms
# keep your current random train transform
train_transform = get_train_transform()  # your existing function

# deterministic, adaptive color for val & test
val_transform  = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")
test_transform = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")


# 📦 Create datasets and dataloaders
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, train_transform)
val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, val_transform)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 👁️ Visual preview
def show_batch(images, masks):
    for i in range(len(images)):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        mask = masks[i][0].cpu().numpy()

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(mask, cmap='gray')
        plt.title("Mask")
        plt.axis('off')
        plt.show()

# Preview one batch
#for images, masks in train_loader:
   # show_batch(images, masks)
  #  break

# Preview one batch
for images, masks in val_loader:
    show_batch(images, masks)
    break



In [ ]:
def check_mask_alignment(dataset, idx=0, alpha=0.5, hflip=True, vflip=False):
    """
    Visual overlay that mirrors the geometry ops in __getitem__.
    - hflip: apply np.fliplr to image & mask (default True to match your __getitem__)
    - vflip: optional vertical flip if you ever need it
    """
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    # --- load raw image ---
    img_id = dataset.image_ids[idx]
    img_info = dataset.coco.loadImgs(img_id)[0]
    img_path = os.path.join(dataset.img_dir, img_info['file_name'])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # --- build raw mask from COCO anns ---
    ann_ids = dataset.coco.getAnnIds(imgIds=img_id)
    anns = dataset.coco.loadAnns(ann_ids)
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, dataset.coco.annToMask(ann))

    # --- apply the SAME geometry as in __getitem__ ---
    if hflip:
        image = np.fliplr(image)
        mask  = np.fliplr(mask)
    if vflip:
        image = np.flipud(image)

    # --- size guard (shouldn’t be needed, but safe) ---
    if mask.shape[:2] != image.shape[:2]:
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    # --- overlay ---
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.imshow(mask, cmap=ListedColormap(['none', 'lime']), alpha=alpha)
    plt.axis("off")
    plt.title("Image + Mask Overlay (geometry-matched to __getitem__)")
    plt.show()


In [ ]:
check_mask_alignment(val_dataset)



In [ ]:
def show_exact_raw(idx, dataset):
    img_id = dataset.image_ids[idx]
    img_info = dataset.coco.loadImgs(img_id)[0]
    img_path = os.path.join(dataset.img_dir, img_info['file_name'])

    # Read raw image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Build raw mask
    ann_ids = dataset.coco.getAnnIds(imgIds=img_id)
    anns = dataset.coco.loadAnns(ann_ids)
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)

    for ann in anns:
        mask = np.maximum(mask, dataset.coco.annToMask(ann))

    # Show raw image and raw mask
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Original Mask")
    plt.axis('off')
    plt.show()

show_exact_raw(0, val_dataset)


In [ ]:
# Updated Model: Use EfficientNet-B3 with no final activation
model_ImNet_Plus = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

In [ ]:
import torch
import torch.nn as nn

# Manual implementation of Focal Tversky Loss
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)  # Apply sigmoid here since activation=None
        targets = targets

        TP = (inputs * targets).sum(dim=(1, 2, 3))
        FP = ((1 - targets) * inputs).sum(dim=(1, 2, 3))
        FN = (targets * (1 - inputs)).sum(dim=(1, 2, 3))

        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma

        return focal_tversky.mean()


In [ ]:
loss_fn = FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75)

# Optimizer
optimizer = torch.optim.Adam(model_ImNet_Plus.parameters(), lr=1e-3)

# Scheduler with fixed closing parenthesis
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Initialize history tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': []
}

# Best score tracking
best_val_dice = 0
patience = 5
epochs_no_improve = 0

# Scheduler for dynamic learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Main training loop
for epoch in range(1, 21):
    model_ImNet_Plus.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model_ImNet_Plus(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_dice += dice_coef(outputs, masks).item()
        train_iou += iou_score(outputs, masks).item()

    model_ImNet_Plus.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model_ImNet_Plus(images)
            val_loss += loss_fn(outputs, masks).item()
            val_dice += dice_coef(outputs, masks).item()
            val_iou += iou_score(outputs, masks).item()

    # Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_train_iou = train_iou / len(train_loader)
    avg_val_iou = val_iou / len(val_loader)

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Logging
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | "
          f"Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f}")

    # Check for improvement
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        epochs_no_improve = 0

        # ✅ Full checkpoint saving
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_ImNet_Plus.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': avg_val_dice,
            'val_loss': avg_val_loss
        }, "best_model.pth")

        print("✅ Saved new best model with optimizer and metrics")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve} epochs")

    # ⛔ Early stopping
    if epochs_no_improve >= patience:
        print("⛔ Early stopping triggered")
        break

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_learning_curves(history)

In [ ]:
# --- visualization ---
def visualize_predictions(model, dataset, device, max_samples=10, thresh=THRESH):
    n = min(len(dataset), max_samples)
    for i in tqdm(range(n), desc="Predicting"):
        image_t, mask_t = dataset[i]            # image_t: (3,H,W) normalized tensor; mask_t: (1,H,W)
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            with eval_autocast():
                logits = model(image_b)
        prob = torch.sigmoid(logits)[0,0].cpu().numpy()
        pred_mask = (prob > thresh).astype(np.uint8)

        img_vis = denormalize_imagenet(image_t)
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        # plots: Original / GT / Pred / Overlay
        fig, axs = plt.subplots(1, 4, figsize=(16, 4))
        axs[0].imshow(img_vis);      axs[0].set_title("Image"); axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray");  axs[1].set_title("Ground Truth"); axs[1].axis("off")
        axs[2].imshow(pred_mask, cmap="gray");axs[2].set_title("Predicted"); axs[2].axis("off")
        overlay = img_vis.copy()
        overlay[ pred_mask.astype(bool) ] = (overlay[ pred_mask.astype(bool) ] * 0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
        axs[3].imshow(overlay);      axs[3].set_title("Overlay"); axs[3].axis("off")
        plt.tight_layout(); plt.show()

# --- run ---
visualize_predictions(model_ImNet_Plus, test_dataset, device, max_samples=10, thresh=THRESH)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Define test transform
test_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE), 
    A.Normalize(),  # Uses ImageNet stats
    ToTensorV2(),
])
test_transform = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")
# 2. Load test dataset
test_dataset = COCOSegmentationDataset(
    img_dir="/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/test/",  
    ann_path="/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/test/_annotations.coco.json", 
    transform=test_transform
)

# 3. Recreate the model with the same architecture used during training
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# 4. Load the full checkpoint and restore the model weights
checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 5. Denormalize function for visualization
def denormalize_image(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    if isinstance(tensor_img, torch.Tensor):
        tensor_img = tensor_img.detach().cpu()
    np_img = tensor_img.permute(1, 2, 0).numpy()
    np_img = np_img * np.array(std) + np.array(mean)
    np_img = np.clip(np_img, 0, 1)
    np_img = (np_img * 255).astype(np.uint8)
    return np_img

# 6. Prediction + Visualization
def visualize_predictions(model, dataset, device, max_samples=None):
    n_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in tqdm(range(n_samples), desc="Predicting"):
        image, true_mask = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            pred_mask = model(image_tensor)

        pred_mask = (pred_mask.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
        image_np = denormalize_image(image)
        true_mask_np = true_mask.squeeze().cpu().numpy()

        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        axs[0].imshow(image_np)
        axs[0].set_title("Original Image")
        axs[1].imshow(true_mask_np, cmap='gray')
        axs[1].set_title("Ground Truth")
        axs[2].imshow(pred_mask, cmap='gray')
        axs[2].set_title("Predicted Mask")
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

# 7. Run predictions
visualize_predictions(model_ImNet_Plus, test_dataset, device, max_samples=20)


In [ ]:
import json, torch, os
import segmentation_models_pytorch as smp

# ---- RELOAD ----
with open("dataset_info.json", "r") as f:
    meta = json.load(f)

# Rebuild transform (no randomness for test)
test_transform = get_val_test_transform_adaptive(
    image_size=meta["image_size"], norm=meta.get("norm", "imagenet")
)

# Rebuild dataset (use same flips you used for val/test)
test_dataset = COCOSegmentationDataset(
    img_dir=meta["img_dir"],
    ann_path=meta["ann_path"],
    transform=test_transform,
    flip_h=meta.get("flip_h", False),
    flip_v=meta.get("flip_v", False),
)

# Rebuild model and load weights
model_ImNet_Plus_restord = smp.UnetPlusPlus(
    encoder_name=meta["model"]["encoder_name"],
    encoder_weights=None,    # no need to redownload
    in_channels=meta["model"]["in_channels"],
    classes=meta["model"]["classes"],
    activation=None
).to(device)
model.load_state_dict(torch.load("temp_model.pth", map_location=device))
model.eval()

print("✅ Restored model & test dataset from disk.")


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import torch
from tqdm import tqdm

def evaluate_confusion_matrix(model, dataset, device, threshold=0.5):
    y_true_all = []
    y_pred_all = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="Evaluating"):
            image, true_mask = dataset[i]
            image_tensor = image.unsqueeze(0).to(device)

            pred_mask = model(image_tensor)
            pred_mask = (pred_mask.squeeze().cpu().numpy() > threshold).astype(np.uint8)

            true_mask_np = true_mask.squeeze().cpu().numpy().astype(np.uint8)

            # Flatten for confusion matrix
            y_true_all.extend(true_mask_np.flatten())
            y_pred_all.extend(pred_mask.flatten())

    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
    return cm

# Run confusion matrix evaluation
cm = evaluate_confusion_matrix(model, test_dataset, device)

print("Confusion Matrix:")
print("TN:", cm[0, 0], "FP:", cm[0, 1])
print("FN:", cm[1, 0], "TP:", cm[1, 1])

# Optional: visualize matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Ground Truth")
plt.show()
